# STAT163 · Practice 1 — what one column holds, what one row means

You will work two practice sessions today and submit the changes in this notebook. 

Move top to bottom. 

Cells marked **Task** are graded on
completion; blocks marked **★ Discuss** and **Going further** are not graded — they are what
the sessions are for: deep exploration of a given topic.

**AI in this practice — recommended mode.** 

Try each task yourself first. 

If you are
stuck in the classroom, ask the teacher or TA. Working
remotely: ask your LLM to explain the concept rather than to solve. Two boundaries are
firm:

- say what you used (the disclosure cell at the end)
- and a notebook authored end to
end by AI is not your work — it is not accepted.

## Before you start

Replace the placeholder with your real name and run the cell.

In [1]:
from selfcheck import check, check_col, check_choice, check_identity

student_name = "Andrii Zakharov"   # ← replace with your name
check_identity(student_name)

✅ student_name: "Andrii Zakharov" saved. Move on.


## Part 0 — The dataset

Real dataset with transactions of a UK-based online retailer: one month (November 2010) of the UCI
**[Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii)** dataset. Every row was produced by an actual sale.

Run the cell to load it.

In [2]:
import pandas as pd

df = pd.read_csv("data/online_retail_2010_11.csv")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,529995,48184,DOORMAT ENGLISH ROSE,6,2010-11-01 08:56:00,7.95,16316.0,United Kingdom
1,529995,48187,DOORMAT NEW ENGLAND,4,2010-11-01 08:56:00,7.95,16316.0,United Kingdom
2,529995,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2010-11-01 08:56:00,6.75,16316.0,United Kingdom
3,529995,22708,WRAP DOLLY GIRL,25,2010-11-01 08:56:00,0.42,16316.0,United Kingdom
4,529995,22781,GUMBALL MAGAZINE RACK,4,2010-11-01 08:56:00,7.65,16316.0,United Kingdom


## Part 1 — First look

The opening ritual for any table you have never seen: how big is it, what are the
columns, how are they stored. Make this the first thing you do with any new table.

*You may need:* `shape`, `len`, `dtypes`, `head`

In [3]:
# Task 1.1 — how many rows does the table have?
n_rows = len(df)

check("n_rows", n_rows, "integer")

✅ n_rows: saved (type — an integer). Move on.


In [4]:
# A free cell for looking around: try df.dtypes, df.head(10), df.sample(5)


## Part 2 — What does each column hold?

From the lecture: every column holds one **kind** of value, and the kind decides what
you can compute.

| Letter | Kind | What it supports |
|---|---|---|
| **N** | Nominal — labels, no order | counting, grouping |
| **O** | Ordinal — ordered, gaps unknown | sort, median |
| **Q** | Quantitative — arithmetic is meaningful | sum, mean, difference |
| **T** | Temporal — points in time | order, range, calendar questions |

The kind is **not** the storage type. Look at the values, not at `df.dtypes`, and
answer with one letter: `"N"`, `"O"`, `"Q"` or `"T"`.

In [5]:
# Task 2.1 — StockCode: 85048, 79323P, 22112… Which kind is it?
kind_stockcode = "N"

check_choice("kind_stockcode", kind_stockcode, {"N", "O", "Q", "T"})

✅ kind_stockcode: answer "N" saved — the check does not test whether it is right. Move on.


In [6]:
# Task 2.2 — Quantity. Which kind is it?
kind_quantity = "Q"

check_choice("kind_quantity", kind_quantity, {"N", "O", "Q", "T"})

✅ kind_quantity: answer "Q" saved — the check does not test whether it is right. Move on.


In [7]:
# Task 2.3 — InvoiceDate. Which kind is it?
kind_invoicedate = "T"

check_choice("kind_invoicedate", kind_invoicedate, {"N", "O", "Q", "T"})

✅ kind_invoicedate: answer "T" saved — the check does not test whether it is right. Move on.


In [8]:
# Task 2.4 — Country. Which kind is it?
kind_country = "N"

check_choice("kind_country", kind_country, {"N", "O", "Q", "T"})

✅ kind_country: answer "N" saved — the check does not test whether it is right. Move on.


**★ Discuss** — which of these questions can this table answer *right now*, and which
column would each one lean on?

1. What was the total revenue in November?
2. Which product has the best customer rating?
3. What share of lines went outside the United Kingdom?
4. Was the last week of November busier than the first?

One of the four cannot be answered at all. Which, and why?

### Missing values — notice, do not fix

*You may need:* `isna`, `sum` — and column names with spaces need the
bracket-and-quotes form: `df["Customer ID"]`.

In [9]:
# Task 2.5 — in how many rows is Customer ID missing?
n_missing_customer = df["Customer ID"].isna().sum()

check("n_missing_customer", n_missing_customer, "integer")

✅ n_missing_customer: saved (type — an integer). Move on.


That is a lot of rows. **★ Discuss:** what kind of purchase could legitimately have no
customer id? Is this an error in the data, or a fact about the shop?

### ★ Predict before you run

The next cell computes `df["Customer ID"].mean()`. Before running it, say what you
expect — out loud, to a neighbor or a TA — and whether the number will mean anything.

In [10]:
df["Customer ID"].mean()

np.float64(15468.172954952024)

The computation succeeds, but the result is the mean of customer *labels* — a number
that answers no question. The storage type allows arithmetic here; the kind of value
does not.

## Part 3 — What is one row?

Finish the sentence for this table: **one row is one ______**.

Write your hypothesis here (double-click this cell, type, then Shift+Enter):

> one row is one …

Now verify it with numbers.

*You may need:* `nunique`, `duplicated`

In [11]:
# Task 3.1 — how many distinct invoices are in the table?
n_invoices = df["Invoice"].nunique()

check("n_invoices", n_invoices, "integer")

✅ n_invoices: saved (type — an integer). Move on.


Compare `n_invoices` with the row count from Task 1.1. Far fewer, or about the same?
(If your number equals the row count, look at your method again.)

What does that comparison rule out about what one row is? Check it against the
hypothesis you wrote above.

Now suppose one row is one *line item*: one product within one invoice. If that were
exactly true, could the pair `(Invoice, StockCode)` ever repeat? The next task checks.

In [12]:
# Task 3.2 — how many rows repeat an (Invoice, StockCode) pair seen earlier in the
# table? Count the later occurrences only, not each pair's first appearance.
n_grain_violations = df.duplicated(subset=["Invoice", "StockCode"]).sum()

check("n_grain_violations", n_grain_violations, "integer")

✅ n_grain_violations: saved (type — an integer). Move on.


What does your count say about the line-item hypothesis? Look at one
`(Invoice, StockCode)` pair up close:

In [13]:
df[(df["Invoice"] == "536162") & (df["StockCode"] == "M")]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76430,536162,M,Manual,8,2010-11-30 12:07:00,1.45,15939.0,United Kingdom
76431,536162,M,Manual,7,2010-11-30 12:07:00,1.45,15939.0,United Kingdom
76450,536162,M,Manual,9,2010-11-30 12:07:00,1.25,15939.0,United Kingdom
76451,536162,M,Manual,2,2010-11-30 12:07:00,1.25,15939.0,United Kingdom
76468,536162,M,Manual,4,2010-11-30 12:07:00,0.72,15939.0,United Kingdom
76469,536162,M,Manual,1,2010-11-30 12:07:00,0.72,15939.0,United Kingdom
76472,536162,M,Manual,1,2010-11-30 12:07:00,1.45,15939.0,United Kingdom


`M` is the shop's code for a *manual* line — a correction typed in by a person.

**★ Discuss:** is this a data error, or a legitimate fact of how the cash register works? What
would you need to know to decide? Note that no pandas method can answer this.

### Exact duplicates

In [14]:
# Task 3.3 — how many rows are exact copies of an earlier row (all 8 columns equal)?
# Hint: duplicated() with no subset compares whole rows.
n_exact_duplicates = df.duplicated().sum()

check("n_exact_duplicates", n_exact_duplicates, "integer")

✅ n_exact_duplicates: saved (type — an integer). Move on.


These rows repeat all eight columns exactly. That could be a scanner registering the item twice, or a
customer buying the same item twice within a minute — the data alone cannot tell you
which. If you ever drop such rows, you write the decision down, because after the drop
the table itself no longer shows that a decision was made.

**The habit from this part** — three one-line checks before any computation: row count,
key uniqueness, duplicate count. This is called *verifying the grain*. Do it on every
table you have not met before.

## Part 4 — The average trap

The owner asks: **"What does a typical order bring in?"**

First, each row's revenue: quantity times price.

In [15]:
# Task 4.1 — add a line_revenue column to df
df["line_revenue"] = df["Quantity"] * df["Price"]

check_col(df, "line_revenue")

✅ ["line_revenue"]: column created. Move on.


In [16]:
# Task 4.2 — the obvious move: the mean of line_revenue
avg_line_revenue = avg_line_revenue = df["line_revenue"].mean()

check("avg_line_revenue", avg_line_revenue, "number")

✅ avg_line_revenue: saved (type — a number). Move on.


Before trusting it, remember Part 3: what does one row stand for? `line_revenue.mean()`
averages over **rows**. Is that what the owner asked about?

You already have everything you need: a grand total, and the invoice count from
Task 3.1. How do you get an average per invoice from those two, with no new methods?
Include all rows for now.

*You may need:* `sum`

In [17]:
# Task 4.3 — the average revenue per invoice
avg_invoice_revenue = df["line_revenue"].sum() / n_invoices

check("avg_invoice_revenue", avg_invoice_revenue, "number")

✅ avg_invoice_revenue: saved (type — a number). Move on.


**★ Discuss:** compare `avg_line_revenue` and `avg_invoice_revenue`. Both are correct
arithmetic on the same column. Which one answers the owner's question — and what
question does the other one answer?

### Negative revenue?

A revenue below zero should stop you. Check whether `line_revenue` has negative
values, find those rows, and look at them closely.

**★ Discuss:** these rows are not all one thing. What distinguishes them? Which
columns tell you?

*You may need:* a boolean filter — `df[...]` with a comparison inside — and `head`.
For text columns: `astype`, the `.str` accessor, `startswith`.

In [18]:
# A free cell for investigating the rows with negative line_revenue


In [19]:
# Task 4.4 — how many rows are cancelled-order lines?
# Decide the criterion yourself, from what you found above, and count the rows
# that match it.
n_cancellations = df["Invoice"].astype(str).str.startswith("C").sum()

check("n_cancellations", n_cancellations, "integer")

✅ n_cancellations: saved (type — an integer). Move on.


In this shop's system, an invoice number that starts with `C` marks a
**cancellation** — those lines have negative `Quantity` and real prices. But not
every negative quantity is a cancellation: some rows carry an ordinary invoice
number, no description and a price of zero — stock corrections of some kind. The
data alone does not say which is which; a person who knows the shop does.

More than one count is defensible here. What matters is that you chose a criterion
and can say what it is.

**★ Discuss:** should cancelled lines count in "what does a typical order bring in"?
No answer is universally right. Make a decision and write it down next to the number
you report.

### ★ Say it in your other language

Count the distinct invoices — Task 3.1 — without pandas:

- if you know **SQL**: say the query out loud, or sketch it on the whiteboard;
- if you know **spreadsheets**: describe how you would get it with a pivot table.

## Going further — optional, ungraded

**The graded part ends here.** The 3 points come from the Task cells above, plus the
disclosure cell below. If you have session time left and want more pandas, pick one
**★ Discuss** question and answer it with code — or take one of these:

- Which weekday was busiest in November? (`InvoiceDate` is stored as text — what has to
  happen before a weekday question becomes answerable?)
- After the United Kingdom, which country appears most? Does counting *rows* answer
  that, or should you count something else?
- Recompute Task 4.3 with cancellations excluded. How much does the answer move?

## Before you submit

**Disclosure.** One line: name the tool and the step, or write "No AI used".

> *Example: "Used Claude to debug the duplicated() call in Task 3.2."*

In [20]:
ai_disclosure = "used Gemini to search for syntax and some data about economics"

check("ai_disclosure", ai_disclosure, "text")

✅ ai_disclosure: saved (type — text). Move on.


**Submit:** restart and re-run the whole notebook (Jupyter: *Kernel → Restart Kernel and
Run All Cells*; Positron: *Run All*), check every cell runs, save, then commit and push.
Finally paste your repository URL into the Week 1 practice assignment on Moodle.

The deadline is shown on the Moodle assignment. Push as often as you like before it.